# 02 — Analyse exploratoire (données réelles)
## Prix des céréales & légumineuses au Sénégal (2007–2026)

Questions : Comment ont évolué les prix réels des céréales ? Quels chocs
(2008, 2022) ? Quelles régions/denrées les plus touchées ? L'inflation
alimentaire suit-elle l'inflation officielle ?

In [1]:

import os, warnings, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (11, 5), "figure.dpi": 110, "axes.titlesize": 13})

PROJ = os.getcwd()
if not os.path.isdir(os.path.join(PROJ, "data")):
    PROJ = os.path.dirname(PROJ)
RAW = os.path.join(PROJ, "data", "raw")
PROC = os.path.join(PROJ, "data", "processed")
GEO = os.path.join(PROJ, "data", "geo")
FIG = os.path.join(PROJ, "reports", "figures")
MODELS = os.path.join(PROJ, "models")
for d in (PROC, FIG, MODELS):
    os.makedirs(d, exist_ok=True)

# Libellés FR des denrées et des régions
COMMOD_FR = {
    "Rice (imported)": "Riz importé (brisé)", "Rice (local)": "Riz local",
    "Rice (ordinary, first quality)": "Riz ordinaire 1re qual.",
    "Rice (ordinary, second quality)": "Riz ordinaire 2e qual.",
    "Millet": "Mil", "Sorghum": "Sorgho", "Sorghum (imported)": "Sorgho importé",
    "Maize (local)": "Maïs local", "Maize (imported)": "Maïs importé",
    "Beans (niebe)": "Niébé (haricot)", "Groundnuts (shelled)": "Arachide décortiquée",
    "Groundnuts (unshelled)": "Arachide en coque",
}
REGION_FR = {"Saint Louis": "Saint-Louis", "Thies": "Thiès",
             "Kedougou": "Kédougou", "Sedhiou": "Sédhiou"}
print("Racine projet :", PROJ)


Racine projet : C:\projet\senegal-food-prices


In [2]:

fact_nat = pd.read_csv(os.path.join(PROC, "fact_prix_national.csv"), parse_dates=["date"])
fact_reg = pd.read_csv(os.path.join(PROC, "fact_prix_regional.csv"), parse_dates=["date"])
panier = pd.read_csv(os.path.join(PROC, "indice_panier_national.csv"), parse_dates=["date"])
panier_reg = pd.read_csv(os.path.join(PROC, "indice_panier_regional.csv"), parse_dates=["date"])
markets = pd.read_csv(os.path.join(PROC, "markets_geo.csv"))
comp = pd.read_csv(os.path.join(PROC, "inflation_compare.csv"))
wb = pd.read_csv(os.path.join(PROC, "worldbank.csv"))
print("OK")


OK


### 1. Évolution des prix nominaux des principales céréales (FCFA/kg)

In [3]:

key = ["Riz importé (brisé)", "Mil", "Maïs local", "Sorgho", "Riz local"]
fig, ax = plt.subplots()
for c in key:
    g = fact_nat[fact_nat["commodity_fr"] == c].sort_values("date")
    ax.plot(g["date"], g["prix_median"], lw=1.5, label=c)
for yr, txt in [(2008, "Crise 2008"), (2022, "Choc 2022")]:
    ax.axvspan(pd.Timestamp(yr,1,1), pd.Timestamp(yr,12,31), color="grey", alpha=.12)
ax.legend(ncol=3, fontsize=8); ax.set_ylabel("Prix médian (FCFA/kg)")
ax.set_title("Prix de détail des céréales de base au Sénégal (réel, WFP)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "01_prix_cereales.png"), bbox_inches="tight")
plt.close(fig); print("→ 01_prix_cereales.png")


→ 01_prix_cereales.png


### 2. Indice du panier céréalier & inflation alimentaire en glissement annuel

In [4]:

fig, ax1 = plt.subplots()
ax1.plot(panier["date"], panier["indice_panier"], color="#1f4e79", lw=2, label="Indice panier (100=2015)")
ax1.set_ylabel("Indice (base 100 = 2015)", color="#1f4e79")
ax2 = ax1.twinx()
ax2.plot(panier["date"], panier["var_annuelle_pct"], color="#c0392b", lw=1.4)
ax2.axhline(0, color="grey", lw=.6); ax2.set_ylabel("Inflation alimentaire a/a (%)", color="#c0392b")
ax1.set_title("Indice du panier céréalier et inflation alimentaire")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "02_indice_panier.png"), bbox_inches="tight")
plt.close(fig); print("→ 02_indice_panier.png")


→ 02_indice_panier.png


### 3. Inflation alimentaire (WFP) vs inflation officielle (Banque mondiale)

In [5]:

c = comp.dropna()
fig, ax = plt.subplots()
ax.plot(c["annee"], c["inflation_alimentaire_WFP_%"], "-o", color="#27ae60", label="Alimentaire (WFP)")
ax.plot(c["annee"], c["inflation_officielle_BM_%"], "-o", color="#1f4e79", label="Officielle (Banque mondiale)")
ax.axhline(0, color="grey", lw=.6); ax.legend(); ax.set_ylabel("%")
ax.set_title("Inflation alimentaire vs inflation officielle au Sénégal")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "03_wfp_vs_officiel.png"), bbox_inches="tight")
plt.close(fig); print("→ 03_wfp_vs_officiel.png  | corr =",
      round(c["inflation_alimentaire_WFP_%"].corr(c["inflation_officielle_BM_%"]), 2))


→ 03_wfp_vs_officiel.png  | corr = 0.91


### 4. Hausse cumulée par denrée (2007 → 2026) et volatilité

In [6]:

g = fact_nat.sort_values("date")
deb = g.groupby("commodity_fr").first()["prix_median"]
fin = g.groupby("commodity_fr").last()["prix_median"]
hausse = ((fin/deb - 1)*100).dropna().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10,6))
ax.barh(hausse.index[::-1], hausse.values[::-1], color=sns.color_palette("flare", len(hausse)))
for i,v in enumerate(hausse.values[::-1]):
    ax.text(v+2, i, f"{v:.0f}%", va="center", fontsize=8)
ax.set_title("Hausse cumulée des prix par denrée (2007 → 2026)"); ax.set_xlabel("%")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "04_hausse_denrees.png"), bbox_inches="tight")
plt.close(fig); print("→ 04_hausse_denrees.png")


→ 04_hausse_denrees.png


### 5. Comparaison régionale — prix médian du riz importé par région (heatmap)

In [7]:

riz = fact_reg[fact_reg["commodity_fr"] == "Riz importé (brisé)"].copy()
riz["annee"] = riz["date"].dt.year
piv = riz.pivot_table(index="region_fr", columns="annee", values="prix_median", aggfunc="median")
piv = piv.loc[:, piv.columns >= 2010]
fig, ax = plt.subplots(figsize=(13,6))
sns.heatmap(piv, cmap="YlOrRd", annot=False, cbar_kws={"label":"FCFA/kg"}, ax=ax)
ax.set_title("Prix médian du riz importé par région et par année"); ax.set_xlabel(""); ax.set_ylabel("")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "05_heatmap_riz_region.png"), bbox_inches="tight")
plt.close(fig); print("→ 05_heatmap_riz_region.png")


→ 05_heatmap_riz_region.png


### 6. Saisonnalité — profil mensuel moyen (soudure)

In [8]:

mil = fact_nat[fact_nat["commodity_fr"].isin(["Mil","Sorgho","Maïs local"])].copy()
mil["mois"] = mil["date"].dt.month
# indice saisonnier = prix / moyenne annuelle de la denrée
mil["an"] = mil["date"].dt.year
mil = mil.merge(mil.groupby(["commodity_fr","an"])["prix_median"].mean().rename("moy_an"),
                on=["commodity_fr","an"])
mil["saison"] = mil["prix_median"]/mil["moy_an"]*100
prof = mil.groupby(["commodity_fr","mois"])["saison"].mean().reset_index()
fig, ax = plt.subplots()
for c,gg in prof.groupby("commodity_fr"):
    ax.plot(gg["mois"], gg["saison"], "-o", label=c, lw=1.5)
ax.axhline(100, color="grey", lw=.6); ax.legend()
ax.set_xticks(range(1,13)); ax.set_xlabel("Mois"); ax.set_ylabel("Indice saisonnier (moy. an = 100)")
ax.set_title("Saisonnalité des céréales locales (pic en période de soudure)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "06_saisonnalite.png"), bbox_inches="tight")
plt.close(fig); print("→ 06_saisonnalite.png")


→ 06_saisonnalite.png


### 7. Carte des marchés (coordonnées GPS réelles) + choroplèthe régionale

In [9]:

from matplotlib.patches import Polygon as MplPoly
import matplotlib.colors as mcolors, matplotlib.cm as cm
geo = json.load(open(os.path.join(GEO, "senegal_regions.geojson"), encoding="utf-8"))
def rings(g): return [g["coordinates"]] if g["type"]=="Polygon" else g["coordinates"]

# indice panier régional récent (12 derniers mois)
last = pd.to_datetime(panier_reg["date"]).max()
recent = panier_reg[pd.to_datetime(panier_reg["date"]) > last - pd.DateOffset(months=12)]
val_reg = recent.groupby("region")["indice_panier"].mean().to_dict()

fig, ax = plt.subplots(figsize=(10,8))
vals=[v for v in val_reg.values()]; norm=mcolors.Normalize(min(vals),max(vals))
sm=cm.ScalarMappable(cmap="YlOrRd",norm=norm); sm.set_array([])
for feat in geo["features"]:
    name=feat["properties"]["shapeName"]; val=val_reg.get(name)
    color=sm.to_rgba(val) if val is not None else "#eeeeee"
    for poly in rings(feat["geometry"]):
        ax.add_patch(MplPoly(np.array(poly[0]), closed=True, facecolor=color, edgecolor="white", lw=.6))
# marchés réels
ax.scatter(markets["longitude"], markets["latitude"], s=18, c="#1f4e79",
           edgecolor="white", linewidth=.4, zorder=5, label="Marchés WFP (64)")
ax.autoscale(); ax.set_aspect("equal"); ax.axis("off"); ax.legend(loc="lower left")
ax.set_title("Indice panier céréalier par région + marchés relevés (réel)")
cbar=fig.colorbar(sm,ax=ax,shrink=.5); cbar.set_label("Indice panier (100=2015)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "07_carte_marches.png"), bbox_inches="tight", dpi=120)
plt.close(fig); print("→ 07_carte_marches.png")


→ 07_carte_marches.png


### 8. Contexte macro (Banque mondiale) — PIB/hab. & production alimentaire vs prix

In [10]:

wbx = wb.copy()
fig, ax1 = plt.subplots()
ax1.plot(panier["date"], panier["indice_panier"], color="#c0392b", lw=2, label="Indice panier céréales")
ax1.set_ylabel("Indice panier (100=2015)", color="#c0392b")
ax2 = ax1.twinx()
if "AG.PRD.FOOD.XD" in wbx.columns:
    ax2.plot(pd.to_datetime(wbx["annee"], format="%Y"), wbx["AG.PRD.FOOD.XD"],
             color="#27ae60", lw=1.8, marker="o", ms=3, label="Prod. alimentaire (BM)")
    ax2.set_ylabel("Indice production alim. (2014-16=100)", color="#27ae60")
ax1.set_title("Prix du panier céréalier vs production alimentaire")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "08_macro_contexte.png"), bbox_inches="tight")
plt.close(fig); print("→ 08_macro_contexte.png")


→ 08_macro_contexte.png


### Synthèse EDA
- Les prix des céréales montrent deux chocs majeurs **réels** : la **crise
  alimentaire de 2008** et le **choc de 2022** (guerre en Ukraine, prix mondiaux).
- L'**inflation alimentaire (WFP)** est nettement corrélée à l'**inflation
  officielle (Banque mondiale)**, ce qui valide la cohérence des données.
- Forte **saisonnalité** des céréales locales (mil, sorgho) : pic en période de
  soudure (avant récoltes).
- Disparités régionales marquées, lisibles sur la carte des 64 marchés réels.
